# 05 — Lenox vs Greenbriar: Neighborhood Income Context

**What are we doing?** We first compare the **host ZCTA** containing each mall, using the *same* ACS 2024 five-year data. Next, we compare the socioeconomic characteristics of **reported origin ZCTAs** using provisional allocated mobility counts.

**Important distinction:** A host ZCTA is not a 3-mile buffer. An origin ZCTA describes a reported device home area, **not** the income of an individual shopper. This notebook is descriptive, not causal. The 2020 CBG crosswalk has not been validated against the Dewey vintage.

Run from the repository root. No restricted Dewey source files are required because the derived, provisional origin totals are already included.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
DATA = Path('results/tables')
FIG = Path('results/figures')
FIG.mkdir(parents=True, exist_ok=True)
acs = pd.read_csv(DATA / 'mall_host_zcta_acs_2024.csv')
acs[['mall','ZCTA','income','B19013_001M','gini','population']]

## 1. Compare mall-host neighborhoods

**Question:** Is the ZCTA containing Lenox more affluent than the ZCTA containing Greenbriar?

We use ACS table **B19013**, median household income, in 2024 inflation-adjusted dollars. `B19013_001M` is the ACS **90% margin of error**, not the standard deviation.

For each host ZCTA, the estimated interval is `estimate ± MOE`. The intervals summarize sampling uncertainty, not all possible geographic mismatch.

In [ ]:
host = acs.set_index('mall')
lenox = host.loc['Lenox Square', 'income']
greenbriar = host.loc['Greenbriar Mall', 'income']
print(f'Income gap: ${lenox-greenbriar:,.0f}')
print(f'Income ratio: {lenox/greenbriar:.2f}x')
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(acs['mall'], acs['income'], yerr=acs['B19013_001M'], capsize=7)
ax.set_ylabel('Median household income (2024 dollars)')
ax.set_title('Host ZCTA income comparison with ACS 90% MOE')
fig.tight_layout()
plt.show()

## 2. Compare Gini, without confusing it with income

Gini measures **within-area income inequality** (roughly 0 = equal, 1 = highly unequal). A lower median income does not automatically mean a higher Gini. These are separate measures.

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(acs['mall'], acs['gini'], yerr=acs['B19083_001M'], capsize=7)
ax.set_ylabel('Gini index')
ax.set_title('Host ZCTA Gini with ACS 90% MOE')
fig.tight_layout()
plt.show()

## 3. Who are the reported origin neighborhoods?

Now change the question: *Which types of home ZCTAs contribute to the observed activity?*

For each mall, define an activity-weighted average of **origin ZCTA median household incomes**:

$$\overline{I}^{(w)}_j = \frac{\sum_i Y_{ij} I_i}{\sum_i Y_{ij}}$$

- $Y_{ij}$: allocated annual reported origin-device counts from ZCTA $i$ to mall area $j$.
- $I_i$: ACS median household income in origin ZCTA $i$.

**This is NOT median income of individual visitors.** A weighted average of area medians does not reconstruct individual incomes. Also, annual counts are not unique visitors.

In [ ]:
flow = pd.read_csv(DATA / 'lenox_greenbriar_origin_income_2020_PROVISIONAL.csv')
valid = flow.dropna(subset=['income','flow']).query('flow > 0').copy()
summary = valid.groupby('Mall').apply(lambda x: pd.Series({
    'allocated_activity': x['flow'].sum(),
    'activity_weighted_origin_income': np.average(x['income'], weights=x['flow']),
    'number_of_origins': x['ZCTA'].nunique()
}), include_groups=False).reset_index()
print(summary.to_string(index=False))

## 4. Visualize the origin-income relationship

Bubble size represents reported allocated origin activity. This plot shows associations and is sensitive to the provisional geographic crosswalk. No bubble represents an identified person.

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
for mall, group in valid.groupby('Mall'):
    ax.scatter(group['income'], group['flow'], s=20+np.sqrt(group['flow'])*2, alpha=.6, label=mall)
ax.set_xlabel('Origin ZCTA median household income (2024 dollars)')
ax.set_ylabel('Allocated annual reported device counts')
ax.legend()
ax.set_title('Origin income and mall-area activity (provisional)')
fig.tight_layout()
plt.show()

## 5. What this does and does not establish

**Supported:** The two host ZCTAs have different ACS median household incomes; reported origin areas have measurable income distributions.

**Not supported:** Greenbriar visitors are necessarily low-income, Lenox visitors are necessarily high-income, or neighborhood income causes mall choice.

**Next research step:** Define equal-radius buffers around both mall coordinates and use a validated 2020 census block / ACS tract allocation to calculate population-weighted context; do not compare a host ZCTA to a three-mile commercial trade area as though they are equivalent. Validate Dewey CBG vintage before publishing definitive origin models.